# Morphological Restoration of rPPG Signals: Comprehensive Research Report
## Project: rPPG_Morphology_Restore (JBHI Scientific Audit)

**Objective**: This report provides the definitive empirical proof for the morphological restoration pipeline. We evaluate the performance of personalized adapters against global models across three high-sampling-rate datasets (CENTAN, Stress 2023, FPS 2023).

### Key Sections:
1. **Statistical Significance**: Global vs. Adapted performance distributions.
2. **Morphological Fidelity**: Inflection Point Area (IPA) and Dicrotic Notch restoration.
3. **Clinical Validation**: Bland-Altman analysis of timing accuracy.
4. **Scientific Integrity**: The 'Smoking Gun' proof against generative hallucination.

In [ ]:
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
from scipy.stats import ttest_rel, wilcoxon, pearsonr
from tqdm.auto import tqdm

# Project Imports
import sys
sys.path.append('.')
from morph_config import RESULTS_DIR, CYCLES_DIR, CHECKPOINTS_DIR, VAE_CKPT_P4, ENCODER_CKPT_P5
from models.vae import PPGVAE
from models.encoder import CameraEncoder
from models.adapter import SubjectAdapter

# Aesthetics
C_GT = '#1A237E'     # Deep Blue
C_RAW = '#E91E63'    # Pink
C_GLOBAL = '#FF9800' # Orange
C_ADAPTED = '#00C853'# Green

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. Data & Model Initialization
We load the Stage 1 VAE (Morphology Prior) and the Stage 2 Encoders (Mapping Models).

In [ ]:
def load_models(enc_name):
    vae_path = os.path.join(CHECKPOINTS_DIR, VAE_CKPT_P4)
    vae = PPGVAE(latent_dim=32).to(device)
    vae.load_state_dict(torch.load(vae_path, map_location=device))
    vae.eval()
    
    in_channels = 1 if enc_name in ['A', 'B'] else 2
    encoder = CameraEncoder(latent_dim=32, in_channels=in_channels, morpho_aux=True).to(device)
    ckpt_name = ENCODER_CKPT_P5.format(name=enc_name)
    encoder_path = os.path.join(CHECKPOINTS_DIR, ckpt_name)
    state_dict = torch.load(encoder_path, map_location=device)
    encoder.load_state_dict(state_dict['encoder'] if 'encoder' in state_dict else state_dict)
    encoder.eval()
    return vae, encoder

vae, encoder_b = load_models('B')
print("Models Loaded Successfully.")